# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and fields

record_sets = metadata.record_sets
print("Available Record Sets:")
record_set_ids = []
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    record_set_ids.append(rs['@id'])

    fields = rs.get('fields', [])
    print("  Fields (@id):")
    for field in fields:
        print(f"    - {field['@id']} | name: {field.get('name', 'N/A')} | type: {field.get('dataType', 'N/A')}")
    print()

# For notebook's next steps, select the first record set id:
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = dict()
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set {record_set_id} with {len(df)} records and columns: {df.columns.tolist()}")

# Show preview for main record set
if main_record_set_id:
    display_cols = dataframes[main_record_set_id].columns.tolist()
    print(f"Columns for {main_record_set_id}:", display_cols)
    dataframes[main_record_set_id].head()
else:
    print("No record set found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes for further analysis.

Below, we demonstrate by selecting a numeric field and a group field from the main record set. Adjust the field `@id` variables as needed.

In [ ]:
from IPython.display import display

# Preview: Display columns for manual field selection
if main_record_set_id:
    print(f"Fields in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())

    # Guess likely candidates for numeric grouping based on field names (adjust these as needed)
    numeric_field = None
    group_field = None
    for col in dataframes[main_record_set_id].columns:
        # Common fields for protein datasets
        if numeric_field is None and ('abundance' in col or 'molecular_weight' in col or 'MW' in col or 'count' in col or 'peptide' in col or 'Coverage' in col):
            # Pick the first likely numeric field
            if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
                numeric_field = col
        if group_field is None and (col.lower() in ('protein_name','description','sample','accession','group','status')):
            group_field = col

    # If couldn't guess, just pick the first for demonstration
    if not numeric_field:
        numeric_field = dataframes[main_record_set_id].select_dtypes(include=['number']).columns[0]
    if not group_field:
        group_field = dataframes[main_record_set_id].columns[0]

    print(f"Using numeric_field: {numeric_field} and group_field: {group_field}")

    # Filtering
    threshold = dataframes[main_record_set_id][numeric_field].mean() if len(dataframes[main_record_set_id]) else 10
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:0.2f}, resulting in {len(filtered_df)} records:")
    display(filtered_df.head())

    # Normalizing
    colnorm = numeric_field + "_normalized"
    filtered_df[colnorm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, colnorm]].head())

    # Grouping
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean shown):")
        display(grouped_df.head())
else:
    print("No numeric or group field candidates found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[main_record_set_id][numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in dataframes[main_record_set_id].columns:
        # Violin plot for numeric field distribution by group, if reasonable
        unique_groups = dataframes[main_record_set_id][group_field].nunique()
        if unique_groups<=20:
            plt.figure(figsize=(10,5))
            sns.violinplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id])
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we successfully loaded and explored the FAIR² dataset on mass spectrometry of extracellular vesicles from stimulated human mast cells using the `mlcroissant` library. 

We demonstrated how to enumerate available record sets and fields by their `@id`, extract data using their `@id`, apply basic filtering and normalization, and visualize distributions. This workflow can be extended for custom analysis and deep exploration using the Croissant metadata structure.*